# 1) My Lane as a ML Task (Type)

**Lane Chosen:** Lane 2: Refresh / Content Opportunity Scoring

### Task Type Definition
This task is structurally a **Learning to Rank (LTR)** and **Scoring** problem framework. While we train an underlying model using probabilistic classification (predicting the likelihood that a page's performance is decaying), our final production output is a **continuous priority score (0-100)**. This score is used to sort the content inventory into a prioritized, descending review queue for human editors.

# 2) Target or Proxy

### The Target Variable
Our ultimate target is **Future Performance Decay / Decline**. 

### The Starter Proxy vs. The Capstone Target
* **Starter Dataset Proxy:** In our initial playground data, we utilize a proxy label derived from the current window: `is_declining_label = (trend_direction == "down")`. This serves as a structural stand-in to build our pipeline end-to-end.
* **Capstone Target Window:** To avoid temporal leakage and create a robust forecasting system, the final production model will utilize a forward-shifted target window:
  `Features (Prior 90 Days) -> Target Status (Next 30 Days Decline)`
  A page is truly flagged if it experiences a sustained drop in organic impressions and average search engine position that crosses our selected policy threshold within that future 30-day window.

# 3) Success Metric

Because this system functions as a decision-support tool for an editorial team with limited manual review capacity, generic accuracy is a poor metric. We care about optimization at the top of our sorted list.

### Core Metrics
* **Precision@K (specifically Precision@50):** Of the top 50 items our system flags as top priority for a refresh, what percentage actually turned out to be genuinely decaying? If an editing team only has capacity to review 50 pages a week, we must ensure those 50 slots contain minimal false alarms.
* **Average Precision (AP) / ROC AUC:** To evaluate the holistic quality of our priority ranking order across the entire dataset slice.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\BS Software Engineering\Internship\repo1\data\raw\content_refresh_anonymized.csv')

unit_of_analysis_sample = df[['content_id', 'client_id', 'impressions_90d', 'content_age_days', 'trend_direction']].head(3)

print("--- UNIT OF ANALYSIS SAMPLE DATAFRAME ---")
print("One row = One unique pseudonymized content item (page) belonging to a specific client over a trailing 90-day window.")
display(unit_of_analysis_sample)

df['target_is_declining'] = (df['trend_direction'] == 'down').astype(int)

print("\n--- TARGET COLUMN DISTRIBUTION SKETCH ---")
print(df['target_is_declining'].value_counts(dropna=False))

--- UNIT OF ANALYSIS SAMPLE DATAFRAME ---
One row = One unique pseudonymized content item (page) belonging to a specific client over a trailing 90-day window.


,content_id,client_id,impressions_90d,content_age_days,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,187,down
1,content_a1fb4e703a9e,client_4e07408562,15320,445,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,141,down



--- TARGET COLUMN DISTRIBUTION SKETCH ---
target_is_declining
1    16262
0    13738
Name: count, dtype: int64


# 5) Why ML Beats a Fixed Rule Here

A traditional business approach relies on a fixed, hard-coded heuristic rule (e.g., `If age > 180 days AND impressions > 500 then flag for refresh`). Machine learning fundamentally outperforms this fixed policy for three reasons:

1. **Non-Linear Feature Interaction:** A fixed rule cannot easily weigh how an increase in a competitor's word count interacts dynamically with a subtle position slip on a high-intent keyword. ML discovers complex, multi-variable boundaries automatically.
2. **Eliminating Arbitrary Thresholds:** Fixed rules suffer from severe edge cases (a page with 499 impressions gets ignored, while 500 triggers an alert). A machine learning model outputs smooth, well-calibrated probabilities.
3. **Adaptability across Clients:** FlyRank deals with highly unbalanced histories across dozens of distinct clients. While a fixed threshold might work well for a massive enterprise site, it completely fails on smaller client sites. ML optimizes parameters dynamically using client patterns, maximizing our Precision@50 across varying scales of organic traffic volume.

# 6) Self-Check

- [x] Names the ML task type (Scoring / Ranking).
- [x] Defines the target/proxy variable clearly.
- [x] Establishes operationally sound success metrics (Precision@K).
- [x] Shows the unit of analysis as an active dataframe in a code cell.
- [x] Explains why this is a true ML problem and not just a hard-coded script.
- [x] Ties the final ranked output directly to a concrete content action.